In [2]:
import pandas as pd
import os

# 1. Get the path to your Windows/Mac Downloads folder automatically
downloads_path = os.path.join(os.path.expanduser("~"), "Downloads")

# 2. Define filenames
file1 = "df_final_demo (1).txt"
file2 = "df_final_experiment_clients.txt"
file3 = "df_final_web_data_pt_1.txt"
file4 = "df_final_web_data_pt_2.txt"

# 3. Load the tables using the full path
try:
    df_demo = pd.read_csv(os.path.join(downloads_path, file1))
    df_exp = pd.read_csv(os.path.join(downloads_path, file2))
    df_web1 = pd.read_csv(os.path.join(downloads_path, file3))
    df_web2 = pd.read_csv(os.path.join(downloads_path, file4))
    

    # 4. Merge Web Data Part 1 and 2 side-by-side
    df_web_side_by_side = pd.merge(
        df_web1, 
        df_web2, 
        on='client_id', 
        how='outer', 
        suffixes=('_web1', '_web2')
    )

    # 5. Merge with Experiment and Demographics
    df_merged = pd.merge(df_web_side_by_side, df_exp, on='client_id', how='left')
    df_final = pd.merge(df_merged, df_demo, on='client_id', how='left')

    # 6. Display the result
    print(f"Final table shape: {df_final.shape}")
    display(df_final.head(100))

except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Make sure the files are inside your Downloads folder.")


Final table shape: (856159, 18)


,client_id,visitor_id_web1,visit_id_web1,process_step_web1,date_time_web1,visitor_id_web2,visit_id_web2,process_step_web2,date_time_web2,Variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,169,201385055_71273495308,749567106_99161211863_557568,confirm,2017-04-12 20:23:09,NaN,NaN,NaN,NaN,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0
1,169,201385055_71273495308,749567106_99161211863_557568,step_3,2017-04-12 20:22:05,NaN,NaN,NaN,NaN,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0
2,169,201385055_71273495308,749567106_99161211863_557568,step_2,2017-04-12 20:20:31,NaN,NaN,NaN,NaN,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0
3,169,201385055_71273495308,749567106_99161211863_557568,step_1,2017-04-12 20:19:45,NaN,NaN,NaN,NaN,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0
4,169,201385055_71273495308,749567106_99161211863_557568,start,2017-04-12 20:19:36,NaN,NaN,NaN,NaN,NaN,21.0,262.0,47.5,M,2.0,501570.72,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,1336,NaN,NaN,NaN,NaN,920624746_32603333901,614001770_19101025926_112779,confirm,2017-05-08 08:23:00,Test,48.0,576.0,42.0,M,4.0,130537.18,6.0,9.0
96,1336,NaN,NaN,NaN,NaN,920624746_32603333901,614001770_19101025926_112779,confirm,2017-05-08 08:21:38,Test,48.0,576.0,42.0,M,4.0,130537.18,6.0,9.0
97,1336,NaN,NaN,NaN,NaN,920624746_32603333901,583743392_96265099036_939815,confirm,2017-05-08 06:08:43,Test,48.0,576.0,42.0,M,4.0,130537.18,6.0,9.0
98,1336,NaN,NaN,NaN,NaN,920624746_32603333901,583743392_96265099036_939815,step_3,2017-05-08 06:06:54,Test,48.0,576.0,42.0,M,4.0,130537.18,6.0,9.0


In [13]:
# General Descriptive Statistics
demo_stats = df_final[['clnt_age', 'clnt_tenure_yr', 'bal', 'num_accts']].describe()

print("--- General Demographic Profile ---")
print(demo_stats)

--- General Demographic Profile ---
            clnt_age  clnt_tenure_yr           bal      num_accts
count  856159.000000   856159.000000  8.561590e+05  856159.000000
mean       31.518240        7.979393  1.142077e+05       1.457374
std        26.911127        8.273774  3.663287e+05       1.179104
min         0.000000        0.000000  0.000000e+00       0.000000
25%         0.000000        0.000000  0.000000e+00       0.000000
50%        33.000000        6.000000  3.714339e+04       2.000000
75%        56.000000       14.000000  9.952008e+04       2.000000
max        96.000000       62.000000  1.632004e+07       8.000000


In [14]:
# Count of clients by gender
gender_counts = df_final['gendr'].value_counts()

# Average balance by gender
balance_by_gender = df_final.groupby('gendr')['bal'].mean()

print("\n--- Gender Analysis ---")
print(gender_counts)
print("\nAverage Balance by Gender:")
print(balance_by_gender)


--- Gender Analysis ---
gendr
0    307915
M    189610
U    179894
F    178723
X        17
Name: count, dtype: int64

Average Balance by Gender:
gendr
0         0.000000
F    151133.944999
M    273787.536231
U    104814.667101
X     27336.672941
Name: bal, dtype: float64


In [15]:
# Create age bins
bins = [0, 25, 40, 60, 100]
labels = ['Gen Z (<25)', 'Millennials (25-40)', 'Gen X (40-60)', 'Seniors (60+)']
df_final['age_group'] = pd.cut(df_final['clnt_age'], bins=bins, labels=labels)

# See which age group logs in the most
logons_by_age = df_final.groupby('age_group', observed=False)['logons_6_mnth'].mean()

print("\n--- Age Group Analysis ---")
print("Average Logons in 6 months by Age Group:")
print(logons_by_age)


--- Age Group Analysis ---
Average Logons in 6 months by Age Group:
age_group
Gen Z (<25)            5.467104
Millennials (25-40)    5.680297
Gen X (40-60)          5.713622
Seniors (60+)          6.561816
Name: logons_6_mnth, dtype: float64
